## Making simulation datasets

In [ ]:
#必要なライブラリのインポート
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import bernoulli
import scanpy as sc
import os


### 各クローンごとのゲノム（リファレンス配列）を準備する

In [ ]:
'''
#各クローンごとにゲノムを設定する
k_n = 2
k = range(0,k_n) #クローンの数（ゲノムの数）
j = 7 #染色体上の、全locusの数
mu = 0.5

r_0 = bernoulli.rvs(mu, size=j, random_state=5)
r_1 = bernoulli.rvs(mu, size=j, random_state=16)

print(r_0)
print(r_1)'


r_0 = np.array([0, 1, 0.5, 0, 0, 1, 0.1])
r_1 = np.array([0, 1, 0.5, 1, 0.1, 0, 0])
dropout_rate = np.array([0.1, 0.1, 0.5, 0.1, 0.1, 0.1, 0.1])'
'''

In [ ]:
#各クローンごとにゲノムを設定する
k_n = 2
k = range(0,k_n) #クローンの数（ゲノムの数）
j = 6 #染色体上の、全locusの数
mu = 0.5
'''
r_0 = bernoulli.rvs(mu, size=j, random_state=5)
r_1 = bernoulli.rvs(mu, size=j, random_state=16)

print(r_0)
print(r_1)'
'''

r_0 = np.array([0, 1, 0, 0, 1, 1])
r_1 = np.array([0, 1, 1, 1, 0, 0])
dropout_rate = np.array([0, 0, 0, 0, 0, 0])

### 各細胞がどのクローンに属するかを決める

In [ ]:
adata = sc.datasets.pbmc3k_processed()

In [ ]:
i = adata.n_obs
#各細胞が、どのクラスタに属しているか
cell_cluster = adata.obs['louvain']
rng = np.random.default_rng(1234)
raw_probs = rng.random((adata.obs['louvain'].nunique(), k_n))
assignment_probs = raw_probs / raw_probs.sum(axis=1)[:, np.newaxis]
cluster_probs = dict(zip(adata.obs['louvain'].cat.categories, assignment_probs.tolist()))


In [ ]:
cluster_probs

In [ ]:
# reference cluster probabilities
cluster_probs['CD4 T cells'] = [1, 0]
cluster_probs['CD14+ Monocytes'] = [0, 1]
cluster_probs['B cells'] = [0.5, 0.5]

In [ ]:
cell_clone = np.zeros((adata.n_obs, k_n))
for l in range(0,adata.n_obs):
    pi_clone_tmp = cluster_probs[cell_cluster[l]]
    cell_clone_tmp = np.random.multinomial(n=1, pvals=pi_clone_tmp, size=1)
    cell_clone[l] = cell_clone_tmp

In [ ]:
cell_clone = []
for l in range(0, adata.n_obs):
    pi_clone_tmp = cluster_probs[cell_cluster[l]]
    cell_clone.append(np.random.choice(k_n, p=pi_clone_tmp))

In [ ]:
cell_clone

### 各細胞ごとに、j番目の上にマッピングされるリードの本数（観測数）と、  mutationを持つリードの本数を計算

In [ ]:
B = np.empty((adata.n_obs,j)) #空の行列（細胞数×locus数）を作成し、ここに保存していく
D = np.random.poisson(lam=80, size=(adata.n_obs,j)) #細胞iの各locus上に、何本リードが乗っているか
for i in range(0,adata.n_obs):
    if cell_clone[i] == 0:
        b_ij_temp = np.random.binomial(D[i], r_0, size=j)
        dropout_mask = np.random.random(j) > dropout_rate
        b_ij_temp = b_ij_temp * dropout_mask
        #b_ij_temp = r_0 * D[i]
        B[i] = b_ij_temp
    elif cell_clone[i] == 1:
        b_ij_temp = np.random.binomial(D[i], r_1, size=j)
        dropout_mask = np.random.random(j) > dropout_rate
        b_ij_temp = b_ij_temp * dropout_mask
        #b_ij_temp = r_1 * D[i]
        B[i] = b_ij_temp

In [ ]:
r_0

In [ ]:
B.max()

In [ ]:
D.max()

In [ ]:
adata_mg = sc.AnnData(X=D, obs=adata.obs, layers={'all_mut':B})
adata_mg.obs['clone'] = cell_clone
adata_mg.obs['clone'] = adata_mg.obs['clone'].astype('category')

In [ ]:
sc.pl.heatmap(adata_mg, layer='all_mut', var_names=['0', '1', '2', '3', '4', '5'] ,show_gene_labels=True, groupby='louvain')

In [ ]:
sc.pl.heatmap(adata_mg, layer='all_mut', var_names=['0', '1', '2', '3', '4', '5'] ,show_gene_labels=True, groupby='clone')

In [ ]:
os.makedirs('../../data/simulation/regenerated', exist_ok=True)
sc.AnnData.write_h5ad(adata, '../../data/simulation/regenerated/adata.h5ad')
sc.AnnData.write_h5ad(adata_mg, '../../data/simulation/regenerated/adata_mg.h5ad')
# Save cluster_probs as JSON instead since it's a dictionary
import json
with open('../../data/simulation/regenerated/cluster_probs.json', 'w') as f:
	json.dump({str(k): v for k, v in cluster_probs.items()}, f)